In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
import timm
from pprint import pprint
model = timm.list_models(pretrained=True)
pprint(model)

In [ ]:
from pprint import pprint

obj = {
    "numbers": list(range(20)),
    "info": {"name": "Alice", "age": 30, "skills": ["py", "ml", "cv"]},
    "text": "hello\nworld"
}

print("==== print(obj) ====")
print(obj)

print("\n==== pprint(obj) ====")
pprint(obj)

# timm 入门示例
本示例将演示如何使用 timm 框架加载预训练模型、进行推理和简单的迁移学习。

In [8]:
# 1. 导入库
import timm
import torch
from torch import nn
from PIL import Image
from torchvision import transforms

In [16]:
# 2. 列出可用的预训练模型
from pprint import pprint
models = timm.list_models('resne*t*')
print(f"可用的预训练模型数量: {len(models)}")
pprint(models)  # 只显示前10个

可用的预训练模型数量: 91
['resnest14d',
 'resnest26d',
 'resnest50d',
 'resnest50d_1s4x24d',
 'resnest50d_4s2x40d',
 'resnest101e',
 'resnest200e',
 'resnest269e',
 'resnet10t',
 'resnet14t',
 'resnet18',
 'resnet18d',
 'resnet26',
 'resnet26d',
 'resnet26t',
 'resnet32ts',
 'resnet33ts',
 'resnet34',
 'resnet34d',
 'resnet50',
 'resnet50_clip',
 'resnet50_clip_gap',
 'resnet50_gn',
 'resnet50_mlp',
 'resnet50c',
 'resnet50d',
 'resnet50s',
 'resnet50t',
 'resnet50x4_clip',
 'resnet50x4_clip_gap',
 'resnet50x16_clip',
 'resnet50x16_clip_gap',
 'resnet50x64_clip',
 'resnet50x64_clip_gap',
 'resnet51q',
 'resnet61q',
 'resnet101',
 'resnet101_clip',
 'resnet101_clip_gap',
 'resnet101c',
 'resnet101d',
 'resnet101s',
 'resnet152',
 'resnet152c',
 'resnet152d',
 'resnet152s',
 'resnet200',
 'resnet200d',
 'resnetaa34d',
 'resnetaa50',
 'resnetaa50d',
 'resnetaa101d',
 'resnetblur18',
 'resnetblur50',
 'resnetblur50d',
 'resnetblur101d',
 'resnetrs50',
 'resnetrs101',
 'resnetrs152',
 'resnetrs200',


In [ ]:

from pprint import pprint
models = timm.list_models(pretrained=True)
print(f"可用的预训练模型数量: {len(models)}")
pprint(models[:10])  # 只显示前10个

In [ ]:
# 3. 加载一个预训练模型（如 resnet18）
model = timm.create_model('resnet18', pretrained=True)
model.eval()  # 设置为推理模式

In [13]:
# 手动下载的预训练模型
model = timm.create_model('resnet50', num_classes=11)#pretrained=True,
print(model.default_cfg)#查看模型cfg

{'url': 'https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-rsb-weights/resnet50_a1_0-14fe96d1.pth', 'hf_hub_id': 'timm/resnet50.a1_in1k', 'architecture': 'resnet50', 'tag': 'a1_in1k', 'custom_load': False, 'input_size': (3, 224, 224), 'test_input_size': (3, 288, 288), 'fixed_input_size': False, 'interpolation': 'bicubic', 'crop_pct': 0.95, 'test_crop_pct': 1.0, 'crop_mode': 'center', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'num_classes': 1000, 'pool_size': (7, 7), 'first_conv': 'conv1', 'classifier': 'fc', 'license': 'apache-2.0', 'origin_url': 'https://github.com/huggingface/pytorch-image-models', 'paper_ids': 'arXiv:2110.00476'}


In [ ]:
# 加载本地下载的 resnet18 或 resnet50 权重
import timm, torch
model_name = 'resnet50'  # 或 'resnet50'
weight_path = f'pretrained_models/{model_name}.pth'  # 按实际文件名修改

model = timm.create_model(model_name, pretrained=False)
state = torch.load(weight_path, map_location='cpu')
if 'state_dict' in state:
    state = state['state_dict']
model.load_state_dict(state, strict=False)
model.eval()

In [28]:
data_cfg = timm.data.resolve_data_config(model.pretrained_cfg)
transform = timm.data.create_transform(**data_cfg)
pprint(transform)


Compose(
    Resize(size=235, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)


In [48]:
# 4. 加载和预处理一张图片（以本地图片为例）
img_path = 'dataset\MILK10k_Training_Input\MILK10k_Training_Input\IL_0000652\ISIC_4671410.jpg'  # 替换为你的图片路径
img = Image.open(img_path).convert('RGB')

input_tensor = transform(img).unsqueeze(0)  # 增加 batch 维度
input_tensor.shape

torch.Size([1, 3, 224, 224])

In [55]:
# 5. 使用模型进行推理
output = model(input_tensor)
print(output.shape)
probs = torch.nn.functional.softmax(output[0], dim=0)
print(probs.shape)
values, indices = torch.topk(probs, k=5)
print(values)
print(indices)


torch.Size([1, 1000])
torch.Size([1000])
tensor([0.2129, 0.1090, 0.0896, 0.0614, 0.0323], grad_fn=<TopkBackward0>)
tensor([ 78, 126, 712, 310, 314])


In [ ]:
IMAGENET_1k_URL = 'https://storage.googleapis.com/bit_models/ilsvrc2012_wordnet_lemmas.txt'
import requests
IMAGENET_1k_LABELS = requests.get(IMAGENET_1k_URL).text.strip().split('\n')
[{'label': IMAGENET_1k_LABELS[idx], 'value': val.item()} for val, idx in zip(values, indices)]

In [ ]:
# 6. 简单的迁移学习（更换最后一层）
num_classes = 2  # 假设二分类任务
model.head = nn.Linear(model.head.in_features, num_classes)
print(model.head)

In [ ]:

import os
import shutil
import tempfile
import matplotlib.pyplot as plt
import PIL
import torch
from torch.utils.tensorboard import SummaryWriter
import numpy as np
from sklearn.metrics import classification_report

from monai.apps import download_and_extract
from monai.config import print_config
from monai.data import decollate_batch, DataLoader
from monai.metrics import ROCAUCMetric
from monai.networks.nets import DenseNet121
from monai.transforms import (
    Activations,
    EnsureChannelFirst,
    AsDiscrete,
    Compose,
    LoadImage,
    RandFlip,
    RandRotate,
    RandZoom,
    ScaleIntensity,
)
from monai.utils import set_determinism

print_config()